# Notebook 07 of 7 — Offline Recording + End-to-End

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

This is the capstone. Six notebooks ago I met the plumbing. Now I want the whole thing to work when the internet doesn't, when a provider is down, and six months from today. This is where `scrape_record` becomes the whole point of the fork — not a workaround, a design.

By the end of this notebook we will be able to answer one question:

> *Can I rebuild my Monday-morning analysis 45 minutes on a plane from a git checkout, and can I do it exactly the same way in six months?*


In [ ]:
# [CODE PLACEHOLDER — Phase B] environment sanity — assert .venv_portfolio is active; STATE dir created; friendly halt with setup command if not


## 1. Why offline

Three failure modes I want to eliminate:

- **Provider outage.** FMP has bad days. Yahoo has bad days. When
  they do, my process should not stop.
- **Rate limits.** Backtests that call live endpoints hit rate limits
  at the exact moment you want to iterate.
- **Reproducibility.** Six months from now when I want to know why I
  placed a trade, "MSFT quote on 2026-07-22" needs to still return the
  same number.

The answer to all three is **snapshots on disk**, checked into git,
served by fetchers that never touch the wire at query time. That is
what `scrape_record` provides for the endpoints without free live
JSON, and what `fmp_cached`'s cassette layer provides for the FMP
endpoints.

## 2. `scrape_record` — the framework

Two commands you'll actually use:

- **`scrape-record record <name> --symbol <SYM>`** — Playwright drives
  the target site (headed or headless), captures the response, writes
  a JSON snapshot to `openbb_platform/tools/scrape_record/snapshots/`.
- **`scrape-record replay <name> --symbol <SYM>`** — reads the JSON,
  passes it through the extractor, returns the same shape the live
  path would have. This is what fetchers call under the hood.

Two auxiliary commands:

- **`scrape-record list`** — every recording available.
- **`scrape-record verify`** — sanity-check every checked-in snapshot
  against its extractor (catches schema drift after a fetcher change).

*The code cell below runs `scrape-record list` and `verify` against
every snapshot in the repo.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] shell out to `scrape-record list` and `scrape-record verify`; print counts + any drift warnings


## 3. Path-traversal defense — a security aside

Symbol strings reach file I/O (`snapshots/<name>/<SYMBOL>.json`).
That's a path-traversal vector if you're not careful. PR #1351 landed
defense-in-depth:

1. **Pydantic pattern** on every fetcher's `symbol` field —
   `^[A-Za-z0-9._\-^=]{1,32}$` — catches slashes and NUL at the API
   layer.
2. **Config-level allowlist** in `scrape_record.config._validate_snapshot_component`
   — rejects `.`, `..`, NUL, and anything outside a 64-char allowlist.
3. **`.resolve()` + `.relative_to()` assertion** in
   `snapshot_path()` — even if the first two are bypassed, the
   resolved path must live inside `snapshots_dir` or the call raises.

You will not run into this in normal use. But it's the shape of
security aside that gets a fork through a review, and it's why the
symbol allowlist looks over-strict.

## 4. `portfolio_export` — the sibling tool

`scrape_record` records **public** data into the repo. `portfolio_export`
is its sibling for **private** data — brokerage exports (Fidelity
positions, Schwab tax lots, and so on). Different rules:

- **`scrape_record` snapshots** live **inside** the repo (public data,
  reproducible builds).
- **`portfolio_export` downloads** live **outside** the repo, at
  `H:\masterswork\browser_exports\` on this machine (private data,
  never committed).

The `portfolio_export` CLI has parallel commands (`record`, `replay`,
`csvs`, `inspect`, `list`, `status`, `tag`) but everything it produces
is gitignored by design. Sam won't run it in this series — it's for the
techtrade lane. But it's why the config-layer distinction exists (see
`portfolio_export/config.py:_validate_outside_repo` vs
`scrape_record/config.py:_validate_snapshots_inside_package`).

## 5. The Monday-morning routine — end to end

Now the capstone. One flow that reloads state from NB01-NB06 and
produces one HTML report. This is what Sam actually runs on a Monday
morning:

1. Confirm environment (from NB01)
2. Deep-dive the most-changed name from Friday close (NB02 pipeline)
3. X-ray + risk on the current basket (NB03)
4. 30-day events + fresh smart-money rollup (NB04)
5. What-if the trade candidates that fall out; paper-trade the ones
   that survive (NB05)
6. Backtest any strategy about to move real capital (NB06)
7. Record any snapshots that got stale (NB07 §2)

Elapsed: about 45 minutes if nothing surprising surfaces. Longer if
NB03 flags concentration that NB05 wasn't planning to address.

*The code cell below runs the whole routine, using the artifacts from
`.notebook_state/`, and writes `portfolio_report.html`.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] orchestrator: reload every .notebook_state/ artifact; walk NB02->NB03->NB04->NB05->NB06 in sequence; compose all outputs into a single portfolio_report.html


## 6. The widget map

The whole engine ships a widget frontend
(`openbb_platform/extensions/portfolio_intel/openbb_portfolio_intel/widget_backend/`)
that the desktop app renders. Every widget is served by one or two
routers you now know:

| Widget (execution-plan milestone) | Router | Introduced in notebook |
|---|---|---|
| Sector look-through pie (M2) | `xray_router.look_through` | NB03 §4 |
| Country look-through pie (M2) | `xray_router.look_through` | NB03 §4 |
| Event calendar timeline (M2) | `events_router` | NB04 §3 |
| Smart-money ribbon (M2) | `smart_money_router.rollup` | NB04 §4 |
| Risk dashboard (M2) | `risk_router.metrics` + `.concentration` | NB03 §6-7 |
| What-if diff (M3) | `portfolio_intel_router` (what-if) | NB05 §3 |
| Brinson attribution (M3) | `portfolio_intel_router` (attribution) | NB05 §5 |
| Paper blotter (M3) | `paper_alerts_router` | NB05 §6 |
| Paper positions (M3) | `paper_alerts_router` | NB05 §6 |
| Paper account summary (M3) | `paper_alerts_router` | NB05 §6 |
| Alerts panel (M4) | `paper_alerts_router.alerts` | NB05 §7 |
| News sentiment (M4) | `news_sentiment_router` | NB04 §6 |
| Backtest-this-portfolio (M4) | `backtest_router` | NB06 §2 |
| Tearsheet (M4) | `backtest.tearsheet` | NB06 §6 |

If you're wondering "where does the code that renders widget X live?"
— it's the router in the middle column. Every widget in the frontend
is one you've called directly from Python in this series.

## 7. Where to go next

The natural directions after this series:

- **`widget_backend/`** — how the routers above serve the frontend.
  The React side lives in `desktop/`.
- **`openbb_platform/extensions/mcp_server/`** — the platform's MCP
  server. Driving it from Claude/Cursor/etc. is a separate guide.
- **Upstream promotion policy.** The `portfolio` branch periodically
  absorbs `develop`; upstream promotion (portfolio → develop) is
  user-initiated only. `CLAUDE.md` at the fork root has the rules.
- **The techtrade lane.** `notebooks/01-06` covers the techtrade
  pipeline — signals, plans, execution — for the same trader from a
  different angle.

## 8. What Sam thinks now (closing)

Six weeks in, running this routine every Monday morning:

- The 10-ticker basket is now 8 tickers. Sam closed AMD and rotated
  half of QQQ into VTV (value tilt) after NB03 kept flagging Tech
  concentration.
- Two paper strategies have been running long enough to have a real
  OOS Sharpe. One has PBO 0.38 and Sam is considering going live at
  1/4 size. The other has PBO 0.71 and got dropped.
- The Sunday sheet is gone. The Monday routine replaced it. It's not
  faster — it's the same 45 minutes — but the routine tells Sam
  things the sheet never did.
- The single largest behavior change: Sam no longer opens positions
  without the NB02 pipeline producing at least "Fair" entry quality.
  That alone cut Sam's turnover by more than half and the
  underperformance vs. SPY closed to about 3 points.

---

## What is NOT in this notebook

- **The MCP client-side walkthrough.** A follow-on guide.
- **The widget-authoring walkthrough** (React side). Lives in `desktop/`, not here.
- **Live-broker adapter authoring.** The paper engine has the shape a real adapter would plug into; the guide for that is future work.

## Preview of the series wrap

You have the whole engine now. There is no NB08 — this is the reunion chapter. When you want to add a widget, extend a router, record a new provider snapshot, or file a bug — start from the corresponding notebook here and follow the router path outward. Every line of the surface has a home in one of these seven files.
